## Vader

In [2]:
!pip install nltk
!pip install pandas
import pandas as pd
#importing and installing the vader libraries
import nltk
nltk.download('vader_lexicon')
#Importing the sentiment analyzer from the vader package
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import statsmodels.api as sm


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/weeb/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [3]:
#reading in the reviews data
file_id = "1npz2M614iA646ldsXwRbOK3Db5RwwecK--h3rIojtRM"
url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=csv"
review_All = pd.read_csv(url)
#Selecting only english elements as vader doesn't read Frenchx
review_Data = review_All[review_All["Language"] == "English"]
#establishing the sentime analyzing object
sia = SentimentIntensityAnalyzer()
#creating an array for the star ratings of the store
Rating = review_Data['Rating']
#creating an array for the text reviews
reviews =review_Data["Review"]
#calculating the vader reviews based on the text
review_scores = reviews.apply(sia.polarity_scores)

#creating the dataframe containing our inforamtion
#compound is the final Vader score
Vader_reviews = pd.DataFrame(
    {"Rating" : Rating,
    "Reviews" : reviews,
     "sentiment score" : review_scores.apply(lambda x: x['compound'])})
print(Vader_reviews)

     Rating                                            Reviews  \
0         5  One of the best supermarkets in the region: it...   
2         1  I've been going to this provigo for 2 decades....   
3         1  All dairy products are a major risk at this gr...   
4         5  Super convenient downtown grocery store with a...   
5         1  The most overpriced and predatory grocery stor...   
..      ...                                                ...   
201       1  If you go here and don't buy on sale, you're g...   
202       5  Great variety of products. Nice prices. We enj...   
204       4  They need better stock control. Often run out ...   
205       2  The produce are never fresh  and it's really h...   
206       3  Meat department is prone to running out of chi...   

     sentiment score  
0             0.6355  
2            -0.0294  
3            -0.7845  
4             0.9337  
5            -0.1761  
..               ...  
201           0.0000  
202           0.8883  


In [4]:
Vader_reviews['sentiment score'].describe()

count    162.000000
mean       0.238749
std        0.649373
min       -0.977400
25%       -0.407575
50%        0.440400
75%        0.842000
max        0.976500
Name: sentiment score, dtype: float64

## Sentistrength

In [5]:
from sentistrength import PySentiStr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [6]:
import pandas as pd
import numpy as np
import nltk

In [7]:
df = pd.read_excel("/Users/weeb/Library/Mobile Documents/com~apple~CloudDocs/*INSY 448/Cases/1/reviews_annotated.xlsx")

In [8]:
df_English = df[df['Language'] == 'English'].copy()

In [9]:
en_senti = PySentiStr()
en_senti.setSentiStrengthPath('/Users/weeb/SentiStrength.jar')
en_senti.setSentiStrengthLanguageFolderPath('/Users/weeb/SentiStrengthDataEnglishOctober2019')


In [10]:
df_English['tokenized'] = df_English.apply(lambda row: nltk.word_tokenize(row['Review']), axis=1)

df_English['position'] = df_English.apply(lambda row: nltk.pos_tag(row['tokenized'], tagset='universal'), axis=1)

In [11]:
## Keeping only Adjectives and adverbs

df_English['Ads_Only'] = df_English.apply(
    lambda row: [word for word in row['position'] if word[1] in ['ADV', 'ADJ']],
    axis=1
)

In [12]:
df_English['Ads_Words'] = df_English.apply(
    lambda row: [word[0] for word in row['Ads_Only']],
    axis=1
)

In [13]:
def safe_sentistrength(words):
    if not isinstance(words, list):
        return None

    text = " ".join(
        w.strip() for w in words
        if isinstance(w, str) and w.strip()
    )

    if not text:
        return 0 #Dealing with no adj

    try:
        return en_senti.getSentiment(text, score="scale")
    except Exception:
        return 0

In [14]:
df_English['Ads_Sentistrength'] = df_English['Ads_Words'].apply(safe_sentistrength)

In [15]:
df_English['Sentistrength'] = df_English['Ads_Sentistrength'].str[0].fillna(0)
df_English.head()

,Unnamed: 0,Name,Rating,Review,Annotat1,Annotat2,Final annotat,Checker,Time Ago,Language,tokenized,position,Ads_Only,Ads_Words,Ads_Sentistrength,Sentistrength
0,0,Juliane Nogueira,5,One of the best supermarkets in the region: it...,Positive,Positive,Positive,NaN,il y a un mois,English,"[One, of, the, best, supermarkets, in, the, re...","[(One, NUM), (of, ADP), (the, DET), (best, ADJ...","[(best, ADJ), (easy, ADJ), (positive, ADJ), (o...","[best, easy, positive, only, not, as, good, la...",[2],2.0
2,2,Sebastien A,1,I've been going to this provigo for 2 decades....,Negative,Negative,Negative,NaN,il y a une semaine,English,"[I, 've, been, going, to, this, provigo, for, ...","[(I, PRON), ('ve, VERB), (been, VERB), (going,...","[(.., ADJ), (n't, ADV), (so, ADV), (almost, AD...","[.., n't, so, almost, bad, consistent, only, n...",[-3],-3.0
3,3,Sven Enters,1,All dairy products are a major risk at this gr...,Negative,Negative,Negative,NaN,il y a 3 semaines,English,"[All, dairy, products, are, a, major, risk, at...","[(All, DET), (dairy, NOUN), (products, NOUN), ...","[(major, ADJ), (how, ADV), (many, ADJ), (ve, A...","[major, how, many, ve, horrible, even, still, ...",[-3],-3.0
4,4,Juliet Martin,5,Super convenient downtown grocery store with a...,Positive,Positive,Positive,NaN,il y a un an,English,"[Super, convenient, downtown, grocery, store, ...","[(Super, NOUN), (convenient, NOUN), (downtown,...","[(fresh, ADJ), (quick, ADJ), (healthy, ADJ), (...","[fresh, quick, healthy, downtown]",[0],0.0
5,5,Marcel,1,The most overpriced and predatory grocery stor...,Negative,Negative,Negative,NaN,il y a un mois,English,"[The, most, overpriced, and, predatory, grocer...","[(The, DET), (most, ADV), (overpriced, ADJ), (...","[(most, ADV), (overpriced, ADJ), (predatory, A...","[most, overpriced, predatory, here, regularly,...",[-4],-4.0


In [16]:
df_English['Sentistrength'].describe()

count    162.000000
mean       0.364198
std        1.775458
min       -4.000000
25%       -1.000000
50%        0.000000
75%        2.000000
max        3.000000
Name: Sentistrength, dtype: float64

## KNN

In [17]:
import pandas as pd

# Extract reviews
file_id = "1npz2M614iA646ldsXwRbOK3Db5RwwecK--h3rIojtRM"
url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=csv"
review_Data = pd.read_csv(url)

# Filter to keep only reviews in english
review_Data = review_Data[review_Data["Language"] == "English"].copy()
review_Data.head()

,Unnamed: 0,Name,Rating,Review,Annotat1,Annotat2,Final annotat,Checker,Time Ago,Language
0,0,Juliane Nogueira,5,One of the best supermarkets in the region: it...,Positive,Positive,Positive,NaN,il y a un mois,English
2,2,Sebastien A,1,I've been going to this provigo for 2 decades....,Negative,Negative,Negative,NaN,il y a une semaine,English
3,3,Sven Enters,1,All dairy products are a major risk at this gr...,Negative,Negative,Negative,NaN,il y a 3 semaines,English
4,4,Juliet Martin,5,Super convenient downtown grocery store with a...,Positive,Positive,Positive,NaN,il y a un an,English
5,5,Marcel,1,The most overpriced and predatory grocery stor...,Negative,Negative,Negative,NaN,il y a un mois,English


In [18]:
Rating = review_Data['Rating']
reviews = review_Data["Review"]

In [19]:
# Install packages for pre-processing
!pip install nltk
import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
nltk.download('stopwords')
stop_words = stopwords.words('english')

# Vectorize and removing the stop words
vectorizer = TfidfVectorizer(stop_words=stop_words)
X = vectorizer.fit_transform(reviews)
y = Rating

[nltk_data] Downloading package stopwords to /Users/weeb/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [61]:
from sklearn.model_selection import train_test_split

#I've edited this

X_train, X_test, y_train, y_test = X[51:], X[0:50], y[51:], y[0:50], 

# train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

In [62]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'cosine'
,metric_params,None
,n_jobs,None


In [63]:
predicted_ratings = knn.predict(X_test)
print(Rating.head())

0    5
2    1
3    1
4    5
5    1
Name: Rating, dtype: int64


In [64]:
# Evaluate model's performance

from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, predicted_ratings))
print(classification_report(y_test, predicted_ratings))

Accuracy: 0.44
              precision    recall  f1-score   support

           1       0.64      0.41      0.50        22
           2       0.50      0.20      0.29         5
           3       0.00      0.00      0.00         3
           4       0.25      0.33      0.29         6
           5       0.56      0.71      0.62        14

    accuracy                           0.44        50
   macro avg       0.39      0.33      0.34        50
weighted avg       0.52      0.44      0.46        50



In [65]:
KNN_results = pd.DataFrame({
    "Actual Rating": y_test.values,
    "Predicted Rating with KNN": predicted_ratings
})

print(KNN_results.head())

   Actual Rating  Predicted Rating with KNN
0              5                          5
1              1                          1
2              1                          3
3              5                          5
4              1                          1


In [66]:
KNN_results['Predicted Rating with KNN'].describe()

count    50.000000
mean      3.280000
std       1.654185
min       1.000000
25%       1.000000
50%       4.000000
75%       5.000000
max       5.000000
Name: Predicted Rating with KNN, dtype: float64

## Multi reg

In [67]:
from sklearn import linear_model

In [68]:
#KNN only has 100

test_df = pd.DataFrame({
    "Review": df_English['Review'].iloc[:50].to_numpy(),
    "Rating": df_English['Rating'].iloc[:50].to_numpy(),
    "KNN": KNN_results['Predicted Rating with KNN'].iloc[:50].to_numpy(),
    "Vader": Vader_reviews['sentiment score'].iloc[:50].to_numpy(),
    "Sentistrength": df_English['Sentistrength'].iloc[:50].to_numpy()
})

test_df 

,Review,Rating,KNN,Vader,Sentistrength
0,One of the best supermarkets in the region: it...,5,5,0.6355,2.0
1,I've been going to this provigo for 2 decades....,1,1,-0.0294,-3.0
2,All dairy products are a major risk at this gr...,1,3,-0.7845,-3.0
3,Super convenient downtown grocery store with a...,5,5,0.9337,0.0
4,The most overpriced and predatory grocery stor...,1,1,-0.1761,-4.0
5,If you’re looking for rotten fruit and rotten ...,1,1,-0.9559,-2.0
6,Its been a years im always buying fruits and v...,1,3,0.7246,2.0
7,"Very good store, I found everything I needed. ...",5,5,0.8012,3.0
8,The cashier and the service is ridiculously ba...,1,1,-0.9224,-2.0
9,"Excellent grocery store, especially for those ...",5,5,0.9589,3.0


In [69]:
test_df['Vader_Normal'] = test_df['Vader']*2 + 3 #To go from -1 to 1 to 1 to 5 range

test_df['Sentistrength_Normal'] = test_df['Sentistrength']*0.5 + 3 #

In [ ]:
import rpy2

In [78]:

%load_ext rpy2.ipython


In [83]:
%%R

library(tidyverse)
library(stats)

In [ ]:
%%R -i test_df

my_lm = lm(Rating ~ KNN + Vader_Normal + Sentistrength_Normal,
        data = test_df)

        summary(my_lm)


Call:
lm(formula = Rating ~ KNN + Vader_Normal + Sentistrength_Normal, 
    data = test_df)

Residuals:
     Min       1Q   Median       3Q      Max 
-2.86079 -0.61828  0.08374  0.52162  2.93105 

Coefficients:
                     Estimate Std. Error t value Pr(>|t|)  
(Intercept)           -1.2082     0.5577  -2.166   0.0355 *
KNN                    0.1542     0.1364   1.131   0.2639  
Vader_Normal           0.3700     0.1806   2.049   0.0462 *
Sentistrength_Normal   0.7139     0.2717   2.627   0.0117 *
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1.22 on 46 degrees of freedom
Multiple R-squared:  0.5454,	Adjusted R-squared:  0.5157 
F-statistic: 18.39 on 3 and 46 DF,  p-value: 5.519e-08

